# RAG-Vorlesungs-Tutor — Client

Dünner Client für das Backend. **Agent, Retrieval und LLM laufen im Server**
(`server/api.py`) und werden beim API-Start aufgebaut. Dieses Notebook lädt nur
PDFs hoch und stellt Fragen – alles über HTTP.

**Server vorher starten** (eigenes Terminal):
```bash
cd server
python api.py
```

## Setup

In [106]:
import requests
from enum import Enum
from pathlib import Path

API_URL = "http://127.0.0.1:8000"


# Chunking-Methoden – muss mit dem Backend-Enum (server/chunking.py) übereinstimmen.
class ChunkingMethod(str, Enum):
    RECURSIVE = "recursive"   # zeichenbasiert, kleine gleichmäßige Chunks
    MARKDOWN = "markdown"     # strukturbasiert (Überschriften), mit Header-Metadaten
    SEMANTIC = "semantic"

# Gewählte Methode für Ingest UND Fragen – einfach hier umstellen:
METHOD = ChunkingMethod.MARKDOWN

## PDFs hochladen (`/ingest`)

Legt die PDFs aus `input_pdfs/` per Upload beim Backend ab; dort werden sie mit
docling in Markdown gewandelt, gechunkt und in die Vektor-DB geschrieben.
Der erste Upload lädt einmalig die docling-Modelle → das kann dauern.

In [105]:
PDF_DIR = Path.cwd() / "input_pdfs"
pdf_paths = sorted(PDF_DIR.glob("*.pdf"))
print(f"{len(pdf_paths)} PDF(s):", [p.name for p in pdf_paths])

if not pdf_paths:
    print("Keine PDFs gefunden – lege welche in", PDF_DIR)
else:
    handles = [open(p, "rb") for p in pdf_paths]
    try:
        files = [("files", (p.name, fh, "application/pdf"))
                 for p, fh in zip(pdf_paths, handles)]
        # formulas=true aktiviert die Formel-/LaTeX-Interpretation (langsamer).
        # Optional zusaetzlich: "ocr": "true" (gescannte PDFs), "delete_pdfs": "false".
        # method waehlt die Chunking-Methode (recursive | markdown).
        resp = requests.post(
            f"{API_URL}/ingest",
            files=files,
            data={"formulas": "true", "method": METHOD.value},
        )
        print("Status:", resp.status_code)
        print(resp.json())
    finally:
        for fh in handles:
            fh.close()

1 PDF(s): ['sample-1.pdf']
Status: 200
{'results': [{'file': 'sample-1.pdf', 'status': "Fehler: Cannot find an appropriate cached snapshot folder for the specified revision on the local disk and outgoing traffic has been disabled. To enable repo look-ups and downloads online, set 'HF_HUB_OFFLINE=0' as environment variable."}], 'deleted_pdfs': 0, 'collection_count': 0}


## Frage stellen (`/ask`)

Schickt die Frage an den Backend-Agenten und zeigt dessen Antwort.

In [86]:
# Das ausführen dieser Zelle setzt die Konversations
messages = []

In [87]:
# Um eine Followup Frage zu stellen nur diese Zelle ausführen.
frage = input("Frage: ")
messages.append({
        "role": "user",
        "content": frage
    })
print(f"{frage}")
resp = requests.post(f"{API_URL}/ask", json={"messages": messages, "method": METHOD.value})
if resp.status_code == 200:
    answer = resp.json()["answer"]
    print("Agent:", answer)

    messages.append({
        "role": "assistant",
        "content": answer
    })
else:
    print("Fehler:", resp.status_code, resp.text)

KeyboardInterrupt: Interrupted by user